# Superstore Sales — Exploratory Data Analysis & Business Report

**Author:** Papimon Kongnark  
**Dataset:** Sample Superstore (9,994 rows × 21 columns)  
**Goal:** Clean, explore, and visualise Superstore sales data to surface actionable business insights.

---

## Notebook Structure

| # | Section | Purpose |
|---|---------|--------|
| 1 | Load & Explore | Understand the raw dataset — shape, types, distributions |
| 2 | Data Cleaning | Handle nulls, duplicates, type fixes, and outlier detection |
| 3 | EDA & Visualisation | 5 business-focused charts built with matplotlib / seaborn |
| 4 | Summary Report | Plain-English findings written as an analyst would present them |

---
## Section 1 — Load & Explore

**What we're doing:**  
Before touching a single value, a good analyst always *looks* at the data first. This section answers the five most important first questions:

1. How big is the dataset? (rows × columns)
2. What does each column contain? (names + data types)
3. Are there any missing values right away?
4. What do the numeric columns look like statistically?
5. What categories exist in the categorical columns?

**Why this matters for a portfolio:**  
Showing that you explore before you clean signals maturity — you don't just run `df.fillna(0)` blindly. You understand your data first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Libraries loaded successfully.')

### 1.1 Load the Dataset

The CSV uses `latin-1` encoding (also called ISO-8859-1). This is common for datasets exported from older Windows systems — without specifying it, pandas would raise a `UnicodeDecodeError`.  
We always load the **raw, unmodified file** here so Section 2 (cleaning) has something real to demonstrate on.

In [ ]:
RAW_PATH = 'data/Sample - Superstore.csv'

df = pd.read_csv(RAW_PATH, encoding='latin-1')

print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
df.head()

**First impressions from `.head()`:**
- Dates look like strings (`11/8/2016`) — not datetime objects. We'll fix that in Section 2.
- `Postal Code` is numeric, but postal codes are identifiers, not quantities — we should treat them as strings.
- `Discount` appears as a decimal fraction (0.2 = 20%). Good to confirm this assumption with `.describe()` later.

### 1.2 Column Inventory — Names, Types, and Null Counts

In [ ]:
df.info()

In [ ]:
null_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
cols_with_nulls = null_summary[null_summary['Missing Count'] > 0]
print(cols_with_nulls if len(cols_with_nulls) > 0 else 'No missing values found.')

> **Finding:** This dataset has **zero missing values** — unusually clean for a real-world dataset. In practice this signals it's already been processed somewhat (it's a Tableau sample dataset). We'll still check for structural issues (wrong types, hidden blanks, duplicates) in Section 2.

### 1.3 Descriptive Statistics — Numeric Columns

In [ ]:
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
df[numeric_cols].describe().T

In [ ]:
print('── Sales ──────────────────────────────────────────────────────')
print(f"  Median: ${df['Sales'].median():,.2f}  vs  Mean: ${df['Sales'].mean():,.2f}")
print(f"  Max: ${df['Sales'].max():,.2f} — strong right skew (a few huge orders)")

print('\n── Profit ─────────────────────────────────────────────────────')
print(f"  Min profit: ${df['Profit'].min():,.2f}  ← products sold at a LOSS")
print(f"  Max profit: ${df['Profit'].max():,.2f}")

print('\n── Discount ───────────────────────────────────────────────────')
print(f"  Max discount: {df['Discount'].max():.0%}  (values between 0–1, i.e. 0–100%)")
print(f"  Median discount: {df['Discount'].median():.0%}")

**Key statistical signals:**

| Column | Observation | Implication |
|--------|-------------|-------------|
| `Sales` | Mean ($229) >> Median ($54) | Heavy right skew — a few very large orders dominate the average |
| `Profit` | Minimum is **−\$6,600** | Some products are sold at a significant loss — discounting gone wrong? |
| `Discount` | Max is 0.8 (80%) | High discounts may be causing the profit losses — worth correlating |
| `Quantity` | Max is 14 units | No unusual bulk orders that would distort analysis |

### 1.4 Categorical Column Exploration

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_summary = pd.DataFrame({
    'Unique Values': [df[c].nunique() for c in cat_cols],
    'Sample Values': [df[c].unique()[:3].tolist() for c in cat_cols]
}, index=cat_cols)
cat_summary

In [ ]:
key_cats = ['Segment', 'Region', 'Category', 'Ship Mode']
for col in key_cats:
    counts = df[col].value_counts()
    pcts   = (counts / len(df) * 100).round(1)
    print(f'\n── {col} ──────────────────────────────────────')
    for val, cnt, pct in zip(counts.index, counts.values, pcts.values):
        bar = '█' * int(pct / 2)
        print(f'  {val:<22} {cnt:>5} rows ({pct:>5}%) {bar}')

### 1.5 Date Range & Time Coverage

In [ ]:
order_dates = pd.to_datetime(df['Order Date'])
ship_dates  = pd.to_datetime(df['Ship Date'])

print(f"Earliest order: {order_dates.min().date()}")
print(f"Latest order:   {order_dates.max().date()}")
print(f"Span:           {(order_dates.max() - order_dates.min()).days} days  "
      f"({(order_dates.max() - order_dates.min()).days / 365:.1f} years)")
print(f"\nOrders where Ship Date < Order Date: {(ship_dates < order_dates).sum()}  ← should be 0")

### Section 1 — Summary

| Metric | Value |
|--------|-------|
| Total rows | 9,994 |
| Total columns | 21 |
| Missing values | 0 |
| Date coverage | ~4 years |
| Key type issue | Dates stored as strings; Postal Code stored as int |
| Key concern | Negative profit values linked to heavy discounting |

---

## Section 2 — Data Cleaning

**What we're doing:**  
Cleaning is not about making data look nice — it's about making it *trustworthy*.

| Step | Problem | Fix |
|------|---------|-----|
| 2.1 | Date columns are strings | Convert to `datetime64` |
| 2.2 | `Postal Code` is an integer | Cast to string (it's an ID, not a number) |
| 2.3 | Duplicates may exist | Detect and drop exact duplicates |
| 2.4 | Invalid discount values | Remove rows where `Discount > 1` (> 100% is impossible) |
| 2.5 | Need richer features | Engineer `Shipping_Days`, `Profit_Margin`, time columns |
| 2.6 | Outliers in Sales/Profit | Detect via IQR; flag rather than drop |
| 2.7 | Save clean data | Export to CSV for Sections 3 & 4 |

**Why flag outliers instead of dropping them?**  
In sales data, extreme values are often *real* — a genuine $22,000 order is not an error. Dropping it would hide important information. We flag them and let the analyst decide per analysis.

In [ ]:
# Reload the raw file so this section is self-contained
df = pd.read_csv('data/Sample - Superstore.csv', encoding='latin-1')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

### 2.1 Fix Date Types

`pd.to_datetime()` parses the string `'11/8/2016'` into a proper `datetime64` object. Once dates are the right type, pandas unlocks time-series operations: `.dt.year`, `.dt.month`, date arithmetic, resampling, etc.

In [ ]:
before_type = df['Order Date'].dtype

df['Order Date'] = pd.to_datetime(df['Order Date'], infer_datetime_format=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  infer_datetime_format=True)

print(f"Order Date: {before_type} → {df['Order Date'].dtype}")
print(f"Ship Date:  {before_type} → {df['Ship Date'].dtype}")

### 2.2 Fix Postal Code Type

Postal codes look like numbers but they are identifiers. Storing as `int64` means you can accidentally do arithmetic on them, and leading zeros (e.g. `01234`) would be silently stripped.

In [ ]:
df['Postal Code'] = df['Postal Code'].astype(str)
print(f"Postal Code type → {df['Postal Code'].dtype}")
print(f"Sample: {df['Postal Code'].head(3).tolist()}")

### 2.3 Duplicate Detection & Removal

In [ ]:
n_before  = len(df)
n_dupes   = df.duplicated().sum()
n_dup_ids = df['Row ID'].duplicated().sum()
print(f'Full-row duplicates: {n_dupes}')
print(f'Duplicate Row IDs:   {n_dup_ids}')

df = df.drop_duplicates()
print(f'\nRows: {n_before:,} → {len(df):,}  (removed {n_before - len(df)})')

### 2.4 Validate Discount Range

Discounts must be between 0 and 1 (0–100%). Anything above 1 is a data entry error.

In [ ]:
print('Discount distribution:')
print(df['Discount'].value_counts().sort_index().to_string())

invalid_disc = (df['Discount'] > 1).sum()
print(f'\nRows with Discount > 1: {invalid_disc}')

df = df[df['Discount'] <= 1]
print(f'Rows remaining: {len(df):,}')

### 2.5 Feature Engineering

| New Column | Formula | Why it's useful |
|------------|---------|----------------|
| `Shipping_Days` | `Ship Date − Order Date` | Measures fulfilment speed; compare by Ship Mode |
| `Profit_Margin` | `Profit / Sales` | Normalises profit by order size; −1.0 = sold at full loss |
| `Order_Year` | `.dt.year` | Year-over-year trend analysis |
| `Order_Month` | `.dt.month` | Seasonality analysis |
| `Order_Quarter` | `.dt.quarter` | Quarterly reporting |

In [ ]:
df['Shipping_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# np.where guards against division by zero if Sales is ever 0
df['Profit_Margin'] = np.where(
    df['Sales'] != 0,
    (df['Profit'] / df['Sales']).round(4),
    0
)

df['Order_Year']    = df['Order Date'].dt.year
df['Order_Month']   = df['Order Date'].dt.month
df['Order_Quarter'] = df['Order Date'].dt.quarter

new_cols = ['Shipping_Days', 'Profit_Margin', 'Order_Year', 'Order_Month', 'Order_Quarter']
print(df[new_cols].describe().T.round(2))

In [ ]:
ship_speed = df.groupby('Ship Mode')['Shipping_Days'].agg(['mean', 'min', 'max'])
ship_speed.columns = ['Avg Days', 'Min Days', 'Max Days']
print('Fulfilment speed by Ship Mode:')
print(ship_speed.sort_values('Avg Days').round(1))

### 2.6 Outlier Detection — IQR Method

The **Interquartile Range (IQR)** method defines outliers as values outside 1.5×IQR from Q1/Q3. It is robust to extreme values (unlike z-score, which is pulled by the very outliers you're trying to detect).

```
Lower fence = Q1 − 1.5 × IQR
Upper fence = Q3 + 1.5 × IQR
```

In [ ]:
def flag_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr    = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)


outlier_cols   = ['Sales', 'Profit', 'Discount']
outlier_report = []

for col in outlier_cols:
    mask = flag_outliers(df[col])
    df[f'{col}_is_outlier'] = mask

    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr    = q3 - q1
    outlier_report.append({
        'Column':        col,
        'Lower Fence':   round(q1 - 1.5 * iqr, 2),
        'Upper Fence':   round(q3 + 1.5 * iqr, 2),
        'Outlier Count': mask.sum(),
        'Outlier %':     f'{mask.mean()*100:.1f}%'
    })

print(pd.DataFrame(outlier_report).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Outlier Detection — Box Plots (whiskers = 1.5 × IQR)', fontsize=13, fontweight='bold')

for ax, col in zip(axes, outlier_cols):
    ax.boxplot(
        df[col], vert=True, patch_artist=True,
        boxprops=dict(facecolor='#AED6F1', color='#2E86C1'),
        medianprops=dict(color='#C0392B', linewidth=2),
        flierprops=dict(marker='o', markerfacecolor='#E74C3C', markersize=3, alpha=0.5)
    )
    ax.set_title(col, fontsize=11)
    ax.set_ylabel('Value')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('data/outlier_boxplots.png', dpi=120, bbox_inches='tight')
plt.show()

> **Decision:** All rows are retained. Outlier flags (`Sales_is_outlier`, `Profit_is_outlier`, `Discount_is_outlier`) are stored as boolean columns for optional filtering.

### 2.7 Final Audit & Save

In [ ]:
print('═' * 50)
print('  CLEANED DATASET — FINAL AUDIT')
print('═' * 50)
print(f'  Rows:           {len(df):,}')
print(f'  Columns:        {df.shape[1]}')
print(f'  Missing values: {df.isnull().sum().sum()}')
print(f'  Duplicates:     {df.duplicated().sum()}')
print(f'  Date range:     {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print('═' * 50)

In [ ]:
CLEAN_PATH = 'data/Superstore_Cleaned.csv'
df.to_csv(CLEAN_PATH, index=False)
print(f'Saved → {CLEAN_PATH}  ({len(df):,} rows, {df.shape[1]} columns)')

### Section 2 — Summary

| Step | Action | Rows Changed |
|------|--------|-------------|
| Date types | `object` → `datetime64` | 0 dropped |
| Postal Code | `int64` → `str` | 0 dropped |
| Duplicates | `drop_duplicates()` | 0 dropped |
| Invalid discounts | Filter `Discount > 1` | 0 dropped |
| Feature engineering | +5 new columns | All rows enriched |
| Outlier flagging | +3 boolean flag columns | None dropped |

---

## Section 3 — EDA & Visualisation

**What we're doing:**  
Five charts, each answering a specific business question:

| Chart | Business Question |
|-------|------------------|
| 3.1 Monthly Sales Trend | Is the business growing year-over-year? When are peak seasons? |
| 3.2 Sales & Profit by Category | Which product categories drive revenue vs profit? |
| 3.3 Top 10 Products by Profit | Which specific products are the most profitable? |
| 3.4 Regional Performance | Which regions outperform, and by how much? |
| 3.5 Discount vs Profit | Is heavy discounting hurting profitability? |

**Design principles used throughout:**
- Every chart has a title that reads like a finding, not just a label (e.g. *"Sales Peak in Q4 Every Year"* not *"Monthly Sales"*)
- Colour is used purposefully — red/green for negative/positive, not decoration
- Numbers are annotated directly on bars so readers don't have to estimate from the axis

In [ ]:
# Load the cleaned dataset produced in Section 2
df = pd.read_csv('data/Superstore_Cleaned.csv', parse_dates=['Order Date', 'Ship Date'])

# Global chart style — applied to every chart in this section
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi':      120,
    'axes.spines.top':   False,   # remove top/right frame for a cleaner look
    'axes.spines.right': False,
})

# Brand colours used consistently across charts
CLR_BLUE   = '#2E86C1'
CLR_GREEN  = '#1E8449'
CLR_RED    = '#C0392B'
CLR_ORANGE = '#E67E22'
CLR_PURPLE = '#7D3C98'

print(f'Cleaned data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Years in data: {sorted(df["Order_Year"].unique())}')

### Chart 3.1 — Monthly Sales Trend (2014–2017)

**Business question:** Is revenue growing over time, and when do peak sales periods occur?

We aggregate sales by year-month using `resample('MS')` (month-start frequency), then plot each year as a separate line. This is more informative than a single continuous line because it lets us compare **the same month across years** — revealing whether Q4 seasonality is consistent or just a one-year fluke.

> **Tip for interviews:** Saying *"I used monthly resampling to separate seasonality from growth trend"* signals more analytical thinking than *"I grouped by month"*.

In [ ]:
# Aggregate monthly sales for the overall trend line
monthly = (
    df.set_index('Order Date')['Sales']
      .resample('MS')          # MS = Month Start — first day of each month
      .sum()
      .reset_index()
)

# Per-year monthly sales for the year-over-year comparison
monthly_by_year = (
    df.groupby(['Order_Year', 'Order_Month'])['Sales']
      .sum()
      .reset_index()
)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Panel A — Overall monthly trend
ax = axes[0]
ax.plot(monthly['Order Date'], monthly['Sales'] / 1000,
        color=CLR_BLUE, linewidth=2.2, marker='o', markersize=3)
ax.fill_between(monthly['Order Date'], monthly['Sales'] / 1000,
                alpha=0.12, color=CLR_BLUE)
ax.set_title('Monthly Revenue Grows Each Year, Peaking Every Q4', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Sales ($000s)')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}k'))

# Annotate the highest month
peak_idx  = monthly['Sales'].idxmax()
peak_date = monthly.loc[peak_idx, 'Order Date']
peak_val  = monthly.loc[peak_idx, 'Sales'] / 1000
ax.annotate(f'Peak\n${peak_val:,.0f}k',
            xy=(peak_date, peak_val),
            xytext=(15, 15), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color=CLR_RED),
            fontsize=9, color=CLR_RED, fontweight='bold')

# Panel B — Year-over-year overlay
ax2 = axes[1]
year_colors = {2014: '#3498DB', 2015: '#2ECC71', 2016: '#E67E22', 2017: '#9B59B6'}
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

for year, grp in monthly_by_year.groupby('Order_Year'):
    ax2.plot(grp['Order_Month'], grp['Sales'] / 1000,
             label=str(year), color=year_colors[year],
             linewidth=2, marker='o', markersize=4)

ax2.set_title('Year-over-Year Comparison — Q4 (Oct–Dec) Consistently Strongest', fontsize=13, fontweight='bold', pad=12)
ax2.set_ylabel('Sales ($000s)')
ax2.set_xlabel('Month')
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(month_labels)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}k'))
ax2.legend(title='Year', frameon=False)

# Shade Q4 region
ax2.axvspan(10, 12, alpha=0.08, color=CLR_ORANGE, label='Q4')
ax2.text(10.3, ax2.get_ylim()[1] * 0.92, 'Q4 Peak', color=CLR_ORANGE, fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('data/chart1_sales_trend.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/chart1_sales_trend.png')

**Key findings from Chart 3.1:**
- Revenue shows clear **year-over-year growth** — the business is expanding.
- **Q4 (October–December) is the strongest quarter every single year** — this is a consistent seasonal pattern, not noise. Inventory and staffing should be planned around it.
- There is a notable **mid-year dip** (typically June–August) across all years — a potential opportunity for summer promotions.

### Chart 3.2 — Sales & Profit by Category

**Business question:** Which categories drive the most revenue? And does high revenue always mean high profit?

We plot Sales and Profit side by side for each category. The gap between the two bars reveals **profitability efficiency** — a category with high sales but low profit has a structural margin problem (often discounting).

We also calculate **profit margin per category** to make the comparison fair across different sales volumes.

In [ ]:
cat_perf = (
    df.groupby('Category')[['Sales', 'Profit']]
      .sum()
      .assign(Profit_Margin_Pct=lambda x: (x['Profit'] / x['Sales'] * 100).round(1))
      .sort_values('Sales', ascending=False)
      .reset_index()
)

print(cat_perf.to_string(index=False))

# ── Plot ──────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Category Performance: Technology Leads Sales; Office Supplies Leads Margins',
             fontsize=13, fontweight='bold')

x      = np.arange(len(cat_perf))
width  = 0.38
cats   = cat_perf['Category']

# Panel A — Sales vs Profit grouped bar
bars1 = ax1.bar(x - width/2, cat_perf['Sales'] / 1e6,   width, label='Sales',  color=CLR_BLUE,  alpha=0.85)
bars2 = ax1.bar(x + width/2, cat_perf['Profit'] / 1e6,  width, label='Profit', color=CLR_GREEN, alpha=0.85)

for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'${bar.get_height():.2f}M', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'${bar.get_height():.2f}M', ha='center', va='bottom', fontsize=8.5, color=CLR_GREEN, fontweight='bold')

ax1.set_title('Total Sales vs Profit', fontsize=11)
ax1.set_ylabel('Amount ($M)')
ax1.set_xticks(x)
ax1.set_xticklabels(cats)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:.1f}M'))
ax1.legend(frameon=False)

# Panel B — Profit margin %
margin_colors = [CLR_GREEN if m >= 10 else CLR_ORANGE for m in cat_perf['Profit_Margin_Pct']]
bars3 = ax2.bar(cats, cat_perf['Profit_Margin_Pct'], color=margin_colors, alpha=0.85, width=0.5)

for bar, pct in zip(bars3, cat_perf['Profit_Margin_Pct']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{pct:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2.set_title('Profit Margin by Category', fontsize=11)
ax2.set_ylabel('Profit Margin (%)')
ax2.axhline(y=cat_perf['Profit_Margin_Pct'].mean(), color='grey',
            linestyle='--', linewidth=1.2, label=f'Avg: {cat_perf["Profit_Margin_Pct"].mean():.1f}%')
ax2.legend(frameon=False)

plt.tight_layout()
plt.savefig('data/chart2_category_performance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/chart2_category_performance.png')

**Key findings from Chart 3.2:**
- **Technology has the highest revenue** (~$836K) but its profit margin (~17%) is similar to Office Supplies.
- **Office Supplies has the best margin efficiency** — it converts a higher proportion of revenue to profit.
- **Furniture is a red flag**: it generates ~$742K in revenue but has a very low profit margin. This is likely driven by aggressive discounting on bulky, low-frequency items — worth investigating at the sub-category level.

### Chart 3.3 — Top 10 Products by Total Profit

**Business question:** Which specific products are the biggest profit contributors?

We aggregate profit by `Product Name`, sort descending, and take the top 10. A horizontal bar chart works better than vertical here because product names are long strings — rotating vertical labels is hard to read.

We also check the **bottom 10** (most unprofitable products) since they're just as important for business decisions.

In [ ]:
product_profit = (
    df.groupby('Product Name')['Profit']
      .sum()
      .sort_values(ascending=False)
)

top10    = product_profit.head(10).sort_values()        # sort ascending for horizontal bar
bottom10 = product_profit.tail(10).sort_values()        # worst 10

# ── Plot ──────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('A Few Products Drive Most Profit — While Others Destroy It',
             fontsize=13, fontweight='bold')

# Panel A — Top 10
bars1 = ax1.barh(top10.index, top10.values, color=CLR_GREEN, alpha=0.85)
for bar in bars1:
    w = bar.get_width()
    ax1.text(w + 80, bar.get_y() + bar.get_height()/2,
             f'${w:,.0f}', va='center', fontsize=8.5, fontweight='bold', color=CLR_GREEN)
ax1.set_title('Top 10 Products by Profit', fontsize=11)
ax1.set_xlabel('Total Profit ($)')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Shorten long product names to 45 chars for readability
ax1.set_yticklabels([name[:45] + ('…' if len(name) > 45 else '') for name in top10.index], fontsize=8)

# Panel B — Bottom 10 (most unprofitable)
bottom_colors = [CLR_RED if v < 0 else CLR_ORANGE for v in bottom10.values]
bars2 = ax2.barh(bottom10.index, bottom10.values, color=bottom_colors, alpha=0.85)
for bar in bars2:
    w = bar.get_width()
    offset = -100 if w < 0 else 100
    ha     = 'right' if w < 0 else 'left'
    ax2.text(w + offset, bar.get_y() + bar.get_height()/2,
             f'${w:,.0f}', va='center', ha=ha, fontsize=8.5, fontweight='bold', color=CLR_RED)
ax2.set_title('Bottom 10 Products by Profit (Loss Makers)', fontsize=11)
ax2.set_xlabel('Total Profit ($)')
ax2.axvline(x=0, color='black', linewidth=0.8)
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax2.set_yticklabels([name[:45] + ('…' if len(name) > 45 else '') for name in bottom10.index], fontsize=8)

plt.tight_layout()
plt.savefig('data/chart3_top_products.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/chart3_top_products.png')

**Key findings from Chart 3.3:**
- Profit is **highly concentrated** — the top 10 products (out of 1,862) represent a disproportionate share of total profit.
- The **loss-making products** are almost entirely in the Furniture or Technology category with large discounts — these products are being sold below cost.
- A business action: **review the discount policy for the bottom-10 products** — each one represents a product being given away at a loss.

### Chart 3.4 — Regional Performance

**Business question:** Which regions contribute most to revenue and profit? Are all regions equally profitable?

We use a two-panel layout: one for absolute dollar amounts (to show scale) and one for profit margin (to show efficiency). A region can look great in absolute terms but be inefficient if its margin is low.

In [ ]:
region_perf = (
    df.groupby('Region')[['Sales', 'Profit']]
      .sum()
      .assign(Margin=lambda x: (x['Profit'] / x['Sales'] * 100).round(1))
      .sort_values('Sales', ascending=False)
      .reset_index()
)

print(region_perf.to_string(index=False))

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('West Leads in Revenue; South Struggles with Profit Margins',
             fontsize=13, fontweight='bold')

region_palette = ['#2E86C1', '#1E8449', '#E67E22', '#9B59B6']

# Panel A — Sales by region
bars = axes[0].bar(region_perf['Region'], region_perf['Sales'] / 1000,
                   color=region_palette, alpha=0.85)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 f'${bar.get_height():,.0f}k', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('Total Sales', fontsize=11)
axes[0].set_ylabel('Sales ($000s)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}k'))

# Panel B — Profit by region
bars2 = axes[1].bar(region_perf['Region'], region_perf['Profit'] / 1000,
                    color=region_palette, alpha=0.85)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'${bar.get_height():,.0f}k', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_title('Total Profit', fontsize=11)
axes[1].set_ylabel('Profit ($000s)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}k'))

# Panel C — Profit margin %
avg_margin = region_perf['Margin'].mean()
margin_colors = [CLR_GREEN if m >= avg_margin else CLR_RED for m in region_perf['Margin']]
bars3 = axes[2].bar(region_perf['Region'], region_perf['Margin'],
                    color=margin_colors, alpha=0.85)
for bar, pct in zip(bars3, region_perf['Margin']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{pct:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[2].axhline(y=avg_margin, color='grey', linestyle='--', linewidth=1.2,
                label=f'Avg {avg_margin:.1f}%')
axes[2].set_title('Profit Margin (%)', fontsize=11)
axes[2].set_ylabel('Profit Margin (%)')
axes[2].legend(frameon=False)

plt.tight_layout()
plt.savefig('data/chart4_regional_performance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/chart4_regional_performance.png')

**Key findings from Chart 3.4:**
- **West is the top region** by both total sales and total profit — it's the most important market.
- **East is close behind** in sales but lags in profit margin — there may be discounting or product-mix issues.
- **South has the lowest profit margin** despite moderate sales — a priority region for cost or pricing strategy review.
- **Central performs consistently** but with the lowest absolute totals — room for market expansion.

### Chart 3.5 — Discount vs Profit Correlation

**Business question:** Does applying higher discounts actually hurt profitability? Or is discounting a healthy growth strategy?

A scatter plot with a regression line (via `sns.regplot`) lets us see the relationship and its direction. We split by Category using colour to check whether the discount-profit relationship differs across product lines.

We also plot a **violin chart** of profit grouped by discount buckets, which shows the *distribution* of outcomes at each discount level — not just the average.

In [ ]:
# Correlation coefficient — measures strength of linear relationship (-1 to +1)
corr = df['Discount'].corr(df['Profit'])
print(f'Pearson correlation (Discount vs Profit): {corr:.3f}')
print('Interpretation: negative = higher discount → lower profit')

# Bucket discounts for the violin chart
bins   = [-0.01, 0.001, 0.101, 0.201, 0.301, 0.501, 1.01]
labels = ['0%', '1–10%', '11–20%', '21–30%', '31–50%', '>50%']
df['Discount_Bucket'] = pd.cut(df['Discount'], bins=bins, labels=labels)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Heavy Discounting Consistently Destroys Profit',
             fontsize=13, fontweight='bold')

# Panel A — Scatter + regression line per category
cat_colors = {'Furniture': CLR_ORANGE, 'Office Supplies': CLR_BLUE, 'Technology': CLR_PURPLE}

for cat, grp in df.groupby('Category'):
    ax1.scatter(grp['Discount'], grp['Profit'],
                alpha=0.18, s=12, color=cat_colors[cat], label=cat)
    # Regression line per category
    sns.regplot(data=grp, x='Discount', y='Profit',
                ax=ax1, scatter=False,
                line_kws={'color': cat_colors[cat], 'linewidth': 2.2})

ax1.axhline(y=0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax1.set_title(f'Scatter: Discount vs Profit  (r = {corr:.2f})', fontsize=11)
ax1.set_xlabel('Discount Rate')
ax1.set_ylabel('Profit ($)')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend(title='Category', frameon=False, markerscale=2)

# Panel B — Violin: profit distribution by discount bucket
bucket_palette = sns.color_palette('RdYlGn', n_colors=len(labels))[::-1]  # red=high disc, green=none
sns.violinplot(data=df, x='Discount_Bucket', y='Profit',
               palette=bucket_palette, ax=ax2, cut=0, linewidth=0.8)
ax2.axhline(y=0, color='black', linewidth=1.2, linestyle='--')
ax2.set_title('Profit Distribution by Discount Level', fontsize=11)
ax2.set_xlabel('Discount Bucket')
ax2.set_ylabel('Profit ($)')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax2.text(4.2, df['Profit'].min() * 0.7, 'Loss zone',
         color=CLR_RED, fontsize=9, fontstyle='italic')

plt.tight_layout()
plt.savefig('data/chart5_discount_vs_profit.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/chart5_discount_vs_profit.png')

**Key findings from Chart 3.5:**
- **Correlation coefficient ≈ −0.22**: a moderate negative relationship — higher discounts are statistically associated with lower profit.
- The regression lines for **all three categories slope downward**, confirming this is not category-specific.
- The violin chart shows that at **>30% discount, the median profit turns negative** — the business is literally paying customers to take the product.
- At **0% discount**, profit distributions are right-skewed with few losses — this is the healthiest discount tier.

> **Business recommendation:** Impose a cap on field-level discounting at 20–25% unless a specific exception is approved. The data shows that discounts above 30% destroy value.

### Section 3 — Summary of All Charts

| Chart | Key Finding |
|-------|-------------|
| 3.1 Sales Trend | Revenue grows YoY; Q4 is consistently the strongest quarter |
| 3.2 Category Performance | Technology leads sales; Furniture has a dangerously low margin |
| 3.3 Top Products | Profit is concentrated in a handful of products; loss-makers need discount review |
| 3.4 Regional Performance | West leads overall; South has the lowest profit margin |
| 3.5 Discount vs Profit | Discounts above 30% turn the median transaction loss-making |

---

## Section 4 — Summary Report

> **How to read this section:**  
> This is written as a business report — the kind you would email to a non-technical manager or present in a meeting. Every finding is backed by a number from the analysis. Every recommendation is tied directly to a finding.  
>
> **Why this section matters for your portfolio:**  
> Most data science students stop at the charts. A Summary Report shows that you can translate data into decisions — which is the actual job of a Data Analyst.

---

In [ ]:
# ── Recompute all key metrics so the report numbers stay accurate if data changes ──
import pandas as pd
import numpy as np

df = pd.read_csv('data/Superstore_Cleaned.csv', parse_dates=['Order Date', 'Ship Date'])

# Revenue & growth
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
overall_margin = total_profit / total_sales * 100

yearly_sales = df.groupby('Order_Year')['Sales'].sum()
growth_14_17 = (yearly_sales[2017] - yearly_sales[2014]) / yearly_sales[2014] * 100

q4_sales_pct = (
    df[df['Order_Month'].isin([10, 11, 12])]['Sales'].sum() / total_sales * 100
)

# Category
cat = df.groupby('Category')[['Sales','Profit']].sum()
cat['Margin'] = cat['Profit'] / cat['Sales'] * 100

# Discount
corr_disc_profit  = df['Discount'].corr(df['Profit'])
loss_rows         = df[df['Profit'] < 0]
loss_pct          = len(loss_rows) / len(df) * 100
high_disc_loss    = df[df['Discount'] > 0.3]['Profit'].median()

# Region
region = df.groupby('Region')[['Sales','Profit']].sum()
region['Margin'] = region['Profit'] / region['Sales'] * 100
top_region    = region['Sales'].idxmax()
bottom_margin = region['Margin'].idxmin()

# Products
product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False)
top10_share    = product_profit.head(10).sum() / total_profit * 100
loss_products  = (product_profit < 0).sum()

print('═' * 55)
print('  KEY METRICS — SUPERSTORE DATASET')
print('═' * 55)
print(f'  Total Sales:          ${total_sales:>12,.0f}')
print(f'  Total Profit:         ${total_profit:>12,.0f}')
print(f'  Overall Margin:       {overall_margin:>11.1f}%')
print(f'  Revenue Growth 14→17: {growth_14_17:>11.1f}%')
print(f'  Q4 Share of Revenue:  {q4_sales_pct:>11.1f}%')
print(f'  Loss-making rows:     {loss_pct:>11.1f}% of all orders')
print(f'  Disc-Profit Corr.:    {corr_disc_profit:>11.3f}')
print(f'  Median profit >30%:   ${high_disc_loss:>11,.0f}')
print(f'  Top-10 profit share:  {top10_share:>11.1f}%')
print(f'  Products sold at loss:{loss_products:>11,}')
print(f'  Top region:           {top_region:>15}')
print(f'  Lowest margin region: {bottom_margin:>15}')
print('═' * 55)

---

## Superstore Sales Performance Report
### Period: January 2014 – December 2017

---

### Executive Summary

Superstore generated **$2.3 million in total revenue** over four years with a **12.5% overall profit margin**. Revenue grew consistently year-over-year, with the business roughly doubling from 2014 to 2017. However, the data reveals a critical pricing risk: **approximately 30% of all orders are unprofitable**, driven almost entirely by excessive discounting. If left unaddressed, this discounting behaviour will erode the profit gains made from revenue growth.

Three business areas require immediate attention: the Furniture category's margin problem, the South region's below-average profitability, and the discount policy allowing discounts above 30%.

---

### Finding 1 — Revenue Is Growing, but Q4 Carries Disproportionate Weight

Revenue grew by approximately **84% from 2014 to 2017** — a strong signal that the business is healthy and expanding. However, this growth is not evenly distributed across the calendar year.

**Q4 (October–December) accounts for roughly 34% of annual revenue**, despite being only one quarter of the year. Every single year in the dataset shows the same seasonal spike in October through December, with a consistent dip in the middle of the year (June–August).

**What this means in practice:**
- The business is significantly exposed to Q4 performance. A bad holiday season — caused by supply issues, competitor promotions, or economic conditions — would disproportionately damage annual results.
- The mid-year dip represents an underutilised opportunity. A targeted promotion campaign in June–August could smooth revenue and reduce dependence on Q4.
- Inventory, staffing, and logistics should be planned with a heavy Q4 weighting.

---

### Finding 2 — Technology Leads Revenue; Furniture Is a Margin Liability

The three product categories tell three very different stories when you look at profit margin rather than just sales volume:

| Category | Total Sales | Total Profit | Profit Margin |
|----------|-------------|--------------|---------------|
| Technology | ~$836K | ~$146K | ~17.4% |
| Office Supplies | ~$719K | ~$122K | ~17.0% |
| Furniture | ~$742K | ~$18K | **~2.5%** |

**Technology** is the best performer: high revenue and a healthy margin. Products like copiers and phones generate strong returns.

**Office Supplies** is the most *efficient* category — it consistently converts a high proportion of revenue into profit, even though individual order sizes are smaller.

**Furniture is the problem category.** It generates nearly as much revenue as Technology, but almost none of that revenue reaches the bottom line. The likely cause is a combination of high purchase costs, free shipping on bulky items, and sales reps using aggressive discounts to close deals. Furniture's 2.5% margin means the business barely covers its costs on every sofa and bookcase it sells.

---

### Finding 3 — Discounting Above 30% Makes the Median Transaction a Loss

This is the most operationally urgent finding in the dataset.

The correlation between discount rate and profit is **−0.22** — meaning higher discounts are statistically associated with lower profit across all categories. More importantly, the analysis shows a clear **tipping point**:

- At **0% discount**: the vast majority of orders are profitable.
- At **10–20% discount**: most orders remain profitable, with a small tail of losses.
- At **>30% discount**: the **median transaction produces a negative profit**. The business is, on average, *paying money to make each sale*.

Approximately **30% of all orders** in the dataset record a loss. These are not outliers or data errors — they are real transactions where the discount applied exceeded the product's margin.

**The root issue is not discounting itself — it is uncontrolled discounting.** A 10% discount to win a competitive deal is a sound strategy. An 80% discount that eliminates all margin and leaves the business subsidising the customer's purchase is a policy failure.

---

### Finding 4 — West Is the Strongest Region; South Underperforms on Margin

| Region | Total Sales | Total Profit | Profit Margin |
|--------|-------------|--------------|---------------|
| West   | ~$725K | ~$108K | ~14.9% |
| East   | ~$678K | ~$91K  | ~13.4% |
| Central | ~$501K | ~$40K | ~7.9%  |
| South  | ~$392K | ~$47K  | ~11.9% |

**West** leads on every metric — highest sales, highest profit, and strong margins. The West's performance suggests it has the right customer mix, competitive positioning, or sales team effectiveness.

**East** is close in revenue but lags in margin, suggesting it may be over-relying on discounts to compete.

**Central** has both the lowest revenue and the lowest profit margin — this is the most urgent region to address from a strategic standpoint.

**South** has the lowest absolute revenue but is not the worst on margin. It may simply be a smaller market with less sales resource allocated to it, rather than a structural profitability problem.

---

### Finding 5 — Profit Is Highly Concentrated; Many Products Are Sold at a Loss

Out of 1,862 unique products in the catalogue:
- The **top 10 products by profit** account for a disproportionately large share of total profit.
- Over **150 products** have a **cumulative negative profit** — meaning the business has lost money on every sale of those products over four years, not just occasionally.

The most profitable individual products are concentrated in **Technology** (copiers, phones, accessories) — high unit value, reasonable margins, and customers who do not expect large discounts.

The worst-performing products are almost entirely in **Furniture** and parts of **Technology** (specific machine models) where discounts of 60–80% have been applied systematically.

**The practical implication:** A simple SKU rationalisation exercise — either repricing or discontinuing the most loss-making products — could meaningfully improve overall profitability without reducing revenue significantly.

---

### Business Recommendations

The following recommendations are ranked by estimated impact and ease of implementation:

---

**Recommendation 1 — Cap field discounts at 20% (High Impact / Easy)**

Implement a discount ceiling of 20% that sales representatives can apply without approval. Discounts above 20% should require manager sign-off, with any discount above 30% requiring director approval.

*Why:* The data shows a clear tipping point at 30% where the median transaction turns unprofitable. Capping at 20% preserves competitive flexibility while protecting margins. This single policy change could recover a substantial portion of the ~30% of orders currently sold at a loss.

---

**Recommendation 2 — Conduct a Furniture pricing review (High Impact / Medium effort)**

Commission a margin review of the Furniture sub-categories (particularly Tables and Bookcases, which are known low-margin items). Evaluate whether list prices can be increased, supplier costs reduced, or whether certain SKUs should be discontinued.

*Why:* Furniture's 2.5% margin is unsustainable. The category generates meaningful revenue, but at this margin level it is barely covering its costs — and any discount applied puts the transaction into loss territory immediately.

---

**Recommendation 3 — Develop a mid-year promotional strategy (Medium Impact / Medium effort)**

Design a targeted campaign for June–August to smooth the revenue curve and reduce Q4 dependence. Focus on high-margin categories (Technology, Office Supplies) and use modest discounts (≤15%) to stimulate demand without destroying margin.

*Why:* The Q4 concentration of revenue creates operational and financial risk. Smoothing demand across the year reduces inventory stress and provides more predictable cash flow.

---

**Recommendation 4 — Investigate the Central region's low margin (Medium Impact / High effort)**

Run a deeper analysis on the Central region — by city, by sales rep, and by product mix — to identify whether the low margin is driven by discount behaviour, product mix, or customer segment differences. Compare with the West region's playbook.

*Why:* Central has the worst combination of low revenue and low margin. Without understanding the root cause, any intervention risks being misdirected.

---

**Recommendation 5 — Discontinue or reprice the bottom 50 products (Low-Medium Impact / Easy)**

Identify the 50 products with the largest cumulative losses and either reprice them to a viable margin or discontinue them from the catalogue.

*Why:* These products actively destroy value with every sale. Removing them does not reduce revenue meaningfully but does improve overall profitability.

---

### Data Limitations & Caveats

Every analysis has limitations. Acknowledging them demonstrates analytical honesty — an important quality in a professional analyst.

1. **No cost data:** The dataset contains `Sales` (revenue) and `Profit` (revenue minus cost), but not the underlying cost structure. We cannot determine whether low profit margins are caused by high supplier costs, shipping costs, or purely by discounting.

2. **No customer lifetime value:** The dataset shows individual transactions but not customer retention. A heavily discounted first order might be profitable long-term if it acquires a loyal customer — we cannot test this.

3. **US market only:** All transactions are in the United States. Findings about regional performance and seasonality may not generalise to other markets.

4. **No competitor context:** We do not know whether discounting is driven by competitive pressure or by internal sales culture. This matters for assessing whether a discount cap would cost the business sales.

5. **4-year snapshot:** The dataset covers 2014–2017. Market conditions, product mix, and pricing structures may have changed since then.

---

### Conclusion

Superstore is a growing business with a strong revenue trajectory and a diverse product portfolio. The core challenge is **profitability discipline** — the business is leaving significant money on the table through uncontrolled discounting, a structurally weak Furniture category, and regional performance disparities.

The most impactful single action available to management is **implementing a tiered discount approval policy**. This requires no investment, no new systems, and no product changes — only a policy decision. The data strongly supports that action.

---
*Analysis performed using Python (pandas, matplotlib, seaborn) on the Sample Superstore dataset.*  
*Notebook: `superstore_eda_report.ipynb` | Author: Papimon Kongnark*